In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Ячейка 1: Импорты и глобальные параметры
import os
import torch
import sys

In [ ]:
import numpy as np
# Хак для совместимости старого кода HRNet с новым NumPy
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float

In [ ]:
# Укажите путь к папке lib внутри репозитория
lib_path = os.path.abspath(r"d:\Libraries\Documents\VScode\Semantic_Segmentation\HRNet_Semantic_Segmentation_HRNet_OCR\lib")

if lib_path not in sys.path:
    sys.path.append(lib_path)

# Теперь импорты внутри библиотеки должны заработать

In [ ]:
# Подключаем наши модули
import dataset
import models_utils
import loss
import model_builder

In [ ]:
# Словарь с конфигурацией обучения
CONFIG = {
    # Пути
    "DATASET_DIR": r"datasets\Jacquard_V2",
    "LOGS_DIR": "./logs",              # Папка для TensorBoard
    "WEIGHTS_DIR": "./weights",
    "PRETRAINED_PATH": r"models\hrnetv2_w48_imagenet_pretrained.pth",
    "YAML_CONFIG_PATH": r"models\seg_hrnet_w48_520x520_sgd_lr2e-2_wd1e-4_bs_16_epoch120.yaml", # Путь к YAML конфигу для HRNet-W48

    # Параметры изображения
    "USE_RESIZE": True,
    "IMG_SIZE": (320, 320),      # (320, 320) (512, 512)
    "MAX_WIDTH": 1024.0,         # Максимально возможная ширина раскрытия гриппера в пикселях (для нормализации)
    
    # Параметры обучения
    "TRAIN_SPLIT": 0.9,          # Доля данных для обучения
    "BATCH_SIZE": 8,             # Для 6GB VRAM начинаем с 1
    "NUM_WORKERS": 4,            # Количество потоков для загрузки данных
    "LEARNING_RATE": 1e-4,
    "EPOCHS": 20,
    "MIXED_PRECISION": True,     # Использование fp16 для экономии памяти

    # Модель
    "MODEL_TYPE": "HRNet-W48"    # Тип модели (HRNet-W48)
}

In [ ]:
# Создаем папки, если их нет
os.makedirs(CONFIG["LOGS_DIR"], exist_ok=True)
os.makedirs(CONFIG["WEIGHTS_DIR"], exist_ok=True)

# Проверка доступности GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

In [ ]:
from model_builder import build_grasp_hrnet
model = build_grasp_hrnet(CONFIG, CONFIG["YAML_CONFIG_PATH"], CONFIG["PRETRAINED_PATH"])
model.to(device)
print("Модель готова!")

In [ ]:
from dataset import get_dataloaders
from loss import GraspLoss
from torch.cuda.amp import GradScaler
from torch.utils.tensorboard import SummaryWriter

# Загрузчики
train_loader, val_loader = get_dataloaders(CONFIG)

# Компоненты обучения
criterion = GraspLoss(lambda_regression=1.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LEARNING_RATE"])
scaler = GradScaler()

# TensorBoard Writer
writer = SummaryWriter(log_dir=CONFIG["LOGS_DIR"])

print(f"Тренировка: {len(train_loader.dataset)} шт. Валидация: {len(val_loader.dataset)} шт.")

Функция "До/После"

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

def visualize_predictions(model, val_loader, device, epoch, writer):
    """Берет один батч из валидации, прогоняет через модель и отправляет картинку в TensorBoard."""
    model.eval()
    
    images, targets = next(iter(val_loader))
    images = images.to(device)
    
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            raw_outputs = model(images)
            outputs = F.interpolate(raw_outputs, size=images.shape[2:], mode='bilinear', align_corners=True)
            
            # Применяем сигмоиду к Quality для получения вероятностей (0..1)
            pred_q = torch.sigmoid(outputs[:, 0, :, :]).cpu().numpy()
            target_q = targets[:, 0, :, :].cpu().numpy()
            
    # Визуализируем первый элемент батча
    img_np = images[0].cpu().permute(1, 2, 0).numpy()
    img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406] # Денормализация
    img_np = np.clip(img_np, 0, 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input RGB")
    
    axes[1].imshow(target_q[0], cmap='jet')
    axes[1].set_title("Ground Truth Quality")
    
    axes[2].imshow(pred_q[0], cmap='jet')
    axes[2].set_title(f"Prediction Quality (Epoch {epoch+1})")
    
    plt.tight_layout()
    
    # Сохраняем в TensorBoard
    writer.add_figure('Validation/Quality_Mask', fig, global_step=epoch)
    plt.close(fig) # Закрываем, чтобы не забивать вывод Jupyter
    model.train()

In [ ]:
import matplotlib.pyplot as plt

def visualize_dataset_samples(dataset, num_samples=3):
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5 * num_samples))
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    titles = ['Original RGB', 'Quality Mask (Q)', 'Angle (Cos/Sin)', 'Width Mask']
    
    for i, idx in enumerate(indices):
        img, target = dataset[idx]
        # img: [3, H, W], target: [4, H, W]
        
        img_np = img.permute(1, 2, 0).numpy()
        # Денормализация для отображения
        img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        img_np = np.clip(img_np, 0, 1)
        
        axes[i, 0].imshow(img_np)
        axes[i, 1].imshow(target[0], cmap='jet') # Качество
        axes[i, 2].imshow(target[1], cmap='twilight') # Угол (Cos или Sin)
        axes[i, 3].imshow(target[3], cmap='viridis') # Ширина
        
        if i == 0:
            for j, title in enumerate(titles):
                axes[i, j].set_title(title)
    
    plt.tight_layout()
    plt.show()

# Запуск визуализации, используя датасет внутри загрузчика
visualize_dataset_samples(train_loader.dataset)

In [ ]:
# 2. Функция потерь и оптимизатор
criterion = GraspLoss(lambda_regression=1.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LEARNING_RATE"])

# 3. Скалер для смешанной точности (Mixed Precision)
# Это наш главный шанс сэкономить память
scaler = GradScaler()

print(f"Подготовка завершена. В датасете {len(train_loader.dataset)} изображений.")

In [ ]:
from tqdm import tqdm # Импортируем шкалу прогресса

best_val_loss = float('inf')


In [ ]:

for epoch in range(CONFIG["EPOCHS"]):
    model.train()
    train_loss = 0
    
    # Оборачиваем наш loader в tqdm для визуализации
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}", unit="batch")
    
    for i, (images, targets) in enumerate(pbar):
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            raw_outputs = model(images)
            outputs = F.interpolate(raw_outputs, size=images.shape[2:], mode='bilinear', align_corners=True)
            loss, loss_dict = criterion(outputs, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        
        # Обновляем информацию в шкале прогресса (справа от полоски)
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}", 
            'q': f"{loss_dict['loss_q']:.3f}", 
            'reg': f"{loss_dict['loss_reg']:.3f}"
        })
        
        # TensorBoard логи
        global_step = epoch * len(train_loader) + i
        writer.add_scalar('Train/Loss_Total', loss.item(), global_step)

    avg_train_loss = train_loss / len(train_loader)
    
    # --- ВАЛИДАЦИЯ ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Validation", leave=False):
            images, targets = images.to(device), targets.to(device)
            with torch.cuda.amp.autocast():
                raw_outputs = model(images)
                outputs = F.interpolate(raw_outputs, size=images.shape[2:], mode='bilinear', align_corners=True)
                loss, _ = criterion(outputs, targets)
                val_loss += loss.item()
                
    avg_val_loss = val_loss / len(val_loader)
    
    # Логи в TensorBoard и визуализация "До/После"
    writer.add_scalar('Epoch/Train_Loss', avg_train_loss, epoch)
    writer.add_scalar('Epoch/Val_Loss', avg_val_loss, epoch)
    visualize_predictions(model, val_loader, device, epoch, writer)
    
    # --- СОХРАНЕНИЕ ---
    # 1. Каждую эпоху для истории
    epoch_save_path = os.path.join(CONFIG["WEIGHTS_DIR"], f"hrnet_epoch_{epoch+1}.pth")
    torch.save({'model_state_dict': model.state_dict()}, epoch_save_path)
    
    # 2. Лучшая модель
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({'model_state_dict': model.state_dict()}, 
                   os.path.join(CONFIG["WEIGHTS_DIR"], "best_grasp_model.pth"))
        print(f"\n[+] Рекорд! Val Loss: {avg_val_loss:.4f}. Модель сохранена.")

In [ ]:
# import torch.nn.functional as F

# # Берем один батч
# images, targets = next(iter(train_loader.dataset))
# images = images.to(device)
# targets = targets.to(device)

# print(f"Входной батч: {images.shape}") # Ожидаем [1, 3, 1024, 1024]

# try:
#     # Прямой проход с использованием Mixed Precision
#     # Прямой проход
#     with torch.cuda.amp.autocast():
#         raw_outputs = model(images) 
    
#         # Растягиваем предсказание [1, 4, 256, 256] -> [1, 4, 1024, 1024]
#         outputs = F.interpolate(raw_outputs, size=images.shape[2:], mode='bilinear', align_corners=True)
    
#         loss, loss_dict = criterion(outputs, targets)
#     # Обратный проход
#     optimizer.zero_grad()
#     scaler.scale(loss).backward()
#     scaler.step(optimizer)
#     scaler.update()
    
#     print(f"Проверка пройдена! Loss: {loss.item():.4f}")
#     print(f"Подробности: {loss_dict}")

# except RuntimeError as e:
#     if "out of memory" in str(e):
#         print("🛑 ОШИБКА: Недостаточно видеопамяти (OOM)!")
#         print("Решение: В первой ячейке CONFIG установите USE_RESIZE = True и IMG_SIZE = (320, 320)")
#         torch.cuda.empty_cache()
#     else:
#         raise e

tensorboard --logdir=./logs

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

def show_model_output(model, val_loader, device, num_samples=3):
    model.eval()
    # Берем один батч из валидации
    images, targets = next(iter(val_loader))
    images = images.to(device)
    
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            # Получаем предсказание
            raw_outputs = model(images)
            # Интерполируем до размера входной картинки (320x320)
            outputs = F.interpolate(raw_outputs, size=images.shape[2:], mode='bilinear', align_corners=True)
            
    # Переводим в numpy для отрисовки
    images = images.cpu()
    targets = targets.cpu().numpy()
    outputs = outputs.cpu().numpy()
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    
    for i in range(num_samples):
        # 1. Оригинальное изображение (денормализация)
        img = images[i].permute(1, 2, 0).numpy()
        img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)
        
        # 2. Маска качества (Ground Truth)
        target_q = targets[i, 0]
        
        # 3. Предсказанная маска качества (пропускаем через сигмоиду)
        pred_q = 1 / (1 + np.exp(-outputs[i, 0])) # sigmoid
        
        # Отрисовка
        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Original RGB")
        
        im1 = axes[i, 1].imshow(target_q, cmap='jet')
        axes[i, 1].set_title("Ground Truth Quality")
        plt.colorbar(im1, ax=axes[i, 1], fraction=0.046, pad=0.04)
        
        im2 = axes[i, 2].imshow(pred_q, cmap='jet')
        axes[i, 2].set_title("Predicted Quality")
        plt.colorbar(im2, ax=axes[i, 2], fraction=0.046, pad=0.04)
        
        for ax in axes[i]: ax.axis('off')

    plt.tight_layout()
    plt.show()

# Запуск визуализации
show_model_output(model, val_loader, device)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

def show_full_grasp_output(model, val_loader, device, num_samples=2):
    model.eval()
    images, targets = next(iter(val_loader))
    images = images.to(device)
    
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            raw_outputs = model(images)
            outputs = F.interpolate(raw_outputs, size=images.shape[2:], mode='bilinear', align_corners=True)
            
    images = images.cpu()
    targets = targets.cpu().numpy()
    outputs = outputs.cpu().numpy()
    
    # Названия колонок для визуализации
    cols = ['RGB', 'Q (GT)', 'Q (Pred)', 'Angle (GT)', 'Angle (Pred)', 'Width (GT)', 'Width (Pred)']
    fig, axes = plt.subplots(num_samples, 7, figsize=(25, 4 * num_samples))
    
    for i in range(num_samples):
        # 1. RGB
        img = images[i].permute(1, 2, 0).numpy()
        img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)
        axes[i, 0].imshow(img)
        
        # 2. Quality (GT & Pred)
        q_gt = targets[i, 0]
        q_pred = 1 / (1 + np.exp(-outputs[i, 0])) # Sigmoid
        axes[i, 1].imshow(q_gt, cmap='jet')
        axes[i, 2].imshow(q_pred, cmap='jet')
        
        # 3. Angle (GT & Pred)
        # Угол вычисляется как 0.5 * atan2(sin, cos)
        angle_gt = 0.5 * np.arctan2(targets[i, 2], targets[i, 1])
        angle_pred = 0.5 * np.arctan2(outputs[i, 2], outputs[i, 1])
        axes[i, 3].imshow(angle_gt, cmap='twilight')
        axes[i, 4].imshow(angle_pred, cmap='twilight')
        
        # 4. Width (GT & Pred)
        w_gt = targets[i, 3]
        w_pred = np.clip(outputs[i, 3], 0, 1) # Ширина нормализована 0-1
        axes[i, 5].imshow(w_gt, cmap='viridis')
        axes[i, 6].imshow(w_pred, cmap='viridis')
        
        # Убираем оси и ставим заголовки
        if i == 0:
            for j, title in enumerate(cols):
                axes[i, j].set_title(title)
        for ax in axes[i]: ax.axis('off')

    plt.tight_layout()
    plt.show()

# Запуск
show_full_grasp_output(model, val_loader, device)

In [ ]:
model.eval()
images, targets = next(iter(val_loader))
with torch.no_grad():
    out = model(images.to(device))
    # Интерполируем до 320
    out = F.interpolate(out, size=(320, 320), mode='bilinear')

print(f"Quality Map - Min: {out[:,0].min().item():.4f}, Max: {out[:,0].max().item():.4f}")
print(f"Angle Cos - Min: {out[:,1].min().item():.4f}, Max: {out[:,1].max().item():.4f}")
print(f"Width Map - Min: {out[:,3].min().item():.4f}, Max: {out[:,3].max().item():.4f}")